# Práctica 3 - Algoritmos Asemble

### Importación de librerías y descarga del dataset

In [1]:
import matplotlib.pyplot as plt
from sklearn import datasets
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
import numpy as np
import os

iris = datasets.load_iris()
x = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names
target = iris.target

### Inicialización de los conjuntos de datos

In [2]:
# Conjunto de datos original
original = pd.DataFrame(data=x, columns=feature_names)
original['target'] = target

# Conjunto de datos estandarizado
x_scaled = StandardScaler().fit_transform(x)
estandarizados = pd.DataFrame(data=x_scaled, columns=feature_names)
estandarizados['target'] = target

# Conjunto de datos normalizado
x_minmax = MinMaxScaler().fit_transform(x)
normalizados = pd.DataFrame(data=x_minmax, columns=feature_names)
normalizados['target'] = target

# Creamos objetos PCA reutilizables
pca_95 = PCA(n_components=0.95)
pca_80 = PCA(n_components=0.80)

# PCA 95% sobre datos originales
x_pca95_original = pca_95.fit_transform(x)
columnas_pca95_original = [f'PC{i+1}' for i in range(x_pca95_original.shape[1])]
originalPCA95 = pd.DataFrame(data=x_pca95_original, columns=columnas_pca95_original)
originalPCA95['target'] = target

# PCA 80% sobre datos originales
x_pca80_original = pca_80.fit_transform(x)
columnas_pca80_original = [f'PC{i+1}' for i in range(x_pca80_original.shape[1])]
originalPCA80 = pd.DataFrame(data=x_pca80_original, columns=columnas_pca80_original)
originalPCA80['target'] = target

# PCA 95% sobre datos estandarizados
x_pca95_estandarizado = pca_95.fit_transform(x_scaled)
columnas_pca95_estandarizado = [f'PC{i+1}' for i in range(x_pca95_estandarizado.shape[1])]
estandarizadoPCA95 = pd.DataFrame(data=x_pca95_estandarizado, columns=columnas_pca95_estandarizado)
estandarizadoPCA95['target'] = target

# PCA 80% sobre datos estandarizados
x_pca80_estandarizado = pca_80.fit_transform(x_scaled)
columnas_pca80_estandarizado = [f'PC{i+1}' for i in range(x_pca80_estandarizado.shape[1])]
estandarizadoPCA80 = pd.DataFrame(data=x_pca80_estandarizado, columns=columnas_pca80_estandarizado)
estandarizadoPCA80['target'] = target

# PCA 95% sobre datos normalizados
x_pca95_normalizado = pca_95.fit_transform(x_minmax)
columnas_pca95_normalizado = [f'PC{i+1}' for i in range(x_pca95_normalizado.shape[1])]
normalizadoPCA95 = pd.DataFrame(data=x_pca95_normalizado, columns=columnas_pca95_normalizado)
normalizadoPCA95['target'] = target

# PCA 80% sobre datos normalizados
x_pca80_normalizado = pca_80.fit_transform(x_minmax)
columnas_pca80_normalizado = [f'PC{i+1}' for i in range(x_pca80_normalizado.shape[1])]
normalizadoPCA80 = pd.DataFrame(data=x_pca80_normalizado, columns=columnas_pca80_normalizado)
normalizadoPCA80['target'] = target

### Creación de particiones para validación cruzada 

In [3]:

# Configuración de validación cruzada
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Diccionario con todos los conjuntos de datos
datasets = {
    'original': original,
    'estandarizados': estandarizados,
    'normalizados': normalizados,
    'originalPCA95': originalPCA95,
    'originalPCA80': originalPCA80,
    'estandarizadoPCA95': estandarizadoPCA95,
    'estandarizadoPCA80': estandarizadoPCA80,
    'normalizadoPCA95': normalizadoPCA95,
    'normalizadoPCA80': normalizadoPCA80
}

# Creamos carpeta para guardar particiones
output_dir = 'particiones_cv'
os.makedirs(output_dir, exist_ok=True)

# Generamos particiones para cada conjunto de datos
for dataset_name, dataset in datasets.items():
    print(f"Generando particiones para {dataset_name}...")
    
    # Creamos una carpeta específica para este dataset
    dataset_dir = os.path.join(output_dir, dataset_name)
    os.makedirs(dataset_dir, exist_ok=True)
    
    # Separamos características y target
    X = dataset.drop('target', axis=1)
    y = dataset['target']
    
    # Generamos las 5 particiones
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        # Creamos una carpeta para este fold
        fold_dir = os.path.join(dataset_dir, f'fold_{fold}')
        os.makedirs(fold_dir, exist_ok=True)
        
        # Separamos train y test
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        # Guardamos en CSV
        X_train.to_csv(os.path.join(fold_dir, 'X_train.csv'), index=False)
        X_test.to_csv(os.path.join(fold_dir, 'X_test.csv'), index=False)
        y_train.to_csv(os.path.join(fold_dir, 'y_train.csv'), index=False, header=True)
        y_test.to_csv(os.path.join(fold_dir, 'y_test.csv'), index=False, header=True)

Generando particiones para original...
Generando particiones para estandarizados...
Generando particiones para normalizados...
Generando particiones para originalPCA95...
Generando particiones para originalPCA80...
Generando particiones para estandarizadoPCA95...
Generando particiones para estandarizadoPCA80...
Generando particiones para normalizadoPCA95...
Generando particiones para normalizadoPCA80...
Generando particiones para estandarizadoPCA80...
Generando particiones para normalizadoPCA95...
Generando particiones para normalizadoPCA80...


### Entrenamiento de los distintos modelos con validación cruzada

In [4]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
import pickle

# Definir los modelos a evaluar
models = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', random_state=42, probability=True),
    'NaiveBayes': GaussianNB(),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# Crear carpeta para guardar modelos y resultados
models_dir = 'modelos_cv'
os.makedirs(models_dir, exist_ok=True)

# Iterar sobre cada conjunto de datos
for dataset_name in datasets.keys():
    # Iterar sobre cada fold
    for fold in range(1, n_splits + 1):
        # Leemos los datos del fold correspondiente
        fold_dir = os.path.join(output_dir, dataset_name, f'fold_{fold}')
        
        # Cargamos datos de entrenamiento y test
        X_train = pd.read_csv(os.path.join(fold_dir, 'X_train.csv'))
        y_train = pd.read_csv(os.path.join(fold_dir, 'y_train.csv')).values.ravel()
        
        # Entrenamos y evaluamos cada modelo
        for model_name, model in models.items():
            # Entrenamos el modelo
            model.fit(X_train, y_train)
            
            # Guardamos el modelo entrenado
            model_path = os.path.join(models_dir, dataset_name, model_name)
            os.makedirs(model_path, exist_ok=True)
            model_file = os.path.join(model_path, f'fold_{fold}.pkl')
            with open(model_file, 'wb') as f:
                pickle.dump(model, f)
